<a href="https://colab.research.google.com/github/carolshayle/Python/blob/main/Progress_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files
import io
import os
import base64 # For embedding images in HTML for PDF
from jinja2 import Environment, FileSystemLoader # For templating HTML

# --- Install necessary libraries for PDF generation ---
print("Installing 'kaleido' for static Plotly image export and 'weasyprint' for HTML to PDF conversion...")
!pip install kaleido weasyprint
print("Installation complete.\n")


Installing 'kaleido' for static Plotly image export and 'weasyprint' for HTML to PDF conversion...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.1/847.1 kB 35.1 MB/s eta 0:00:00
Installation complete.



In [2]:
# 1. Upload the Excel file
print("Please upload your 'Status_Report.xlsx' file:")
uploaded = files.upload()

# Read the Excel file into a pandas DataFrame
file_name = next(iter(uploaded))
df = pd.read_excel(io.BytesIO(uploaded[file_name]))

print("\n--- Original DataFrame Head ---")
print(df.head())
print("\n--- DataFrame Info ---")
df.info()

# --- Data Cleaning/Preparation ---
# Ensure 'TOTAL_DONE' is numeric, coercing errors to NaN and then filling with 0
df['TOTAL_DONE'] = pd.to_numeric(df['TOTAL_DONE'], errors='coerce').fillna(0)

# Ensure 'STATUS' categories are consistent for better grouping and plotting
df['STATUS'] = df['STATUS'].astype(str).str.capitalize().replace({
    'Done': 'Done',
    'Ongoing': 'Ongoing',
    'Not started': 'Not Started'
})

# Define colors for status categories for consistent plotting
status_colors = {
    'Done': 'green',
    'Ongoing': 'orange',
    'Not Started': 'red'
}

# --- Initialize dictionaries to store all figures, tables, and summaries for easy export ---
output_figures = {}
output_tables = {}
final_summaries = {}


Please upload your 'Status_Report.xlsx' file:


Saving Status_Report.xlsx to Status_Report.xlsx

--- Original DataFrame Head ---
    DISTRICT SUBDISTRICT STATUS    Assigned  TOTAL_DONE
0  Abdiaziiz  Dhagaxtuur   DONE  Abdisalaan       200.0
1  Abdiaziiz     Gaarisa   DONE  Abdisalaan       124.0
2  Abdiaziiz  Looyacadde   DONE  Abdisalaan       202.0
3  Abdiaziiz       Neero   DONE  Abdisalaan       352.0
4  Boondheer    Daljirka   DONE     Sabrina       251.0

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DISTRICT     93 non-null     object 
 1   SUBDISTRICT  93 non-null     object 
 2   STATUS       93 non-null     object 
 3   Assigned     65 non-null     object 
 4   TOTAL_DONE   65 non-null     float64
dtypes: float64(1), object(4)
memory usage: 3.8+ KB


In [4]:
# 2. Bar graph for status per district with distinct colors
print("\n--- Bar Graph: Status Count per District ---")
status_district_counts = df.groupby(['DISTRICT', 'STATUS']).size().unstack(fill_value=0)

fig_status_district = go.Figure()
for status in ['Done', 'Ongoing', 'Not Started']:
    if status in status_district_counts.columns:
        fig_status_district.add_trace(go.Bar(
            x=status_district_counts.index,
            y=status_district_counts[status],
            name=status,
            marker_color=status_colors.get(status)
        ))

fig_status_district.update_layout(
    barmode='stack',
    title_text='Status Count per District',
    xaxis_title="District",
    yaxis_title="Number of Sub-districts/Items",
    legend_title="Status",
    hovermode="x unified"
)
fig_status_district.show()
output_figures['status_count_per_district'] = fig_status_district


--- Bar Graph: Status Count per District ---


/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [5]:
# 3. Table showing status per subdistrict
print("\n--- Table: Status per Subdistrict ---")
subdistrict_status_table = df[['DISTRICT', 'SUBDISTRICT', 'STATUS']].sort_values(by=['DISTRICT', 'SUBDISTRICT']).reset_index(drop=True)
print(subdistrict_status_table.to_string())

# Display as an interactive Plotly table
fig_table_sub = go.Figure(data=[go.Table(
    header=dict(values=list(subdistrict_status_table.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[subdistrict_status_table.DISTRICT, subdistrict_status_table.SUBDISTRICT, subdistrict_status_table.STATUS],
               fill_color='lavender',
               align='left'))
])
fig_table_sub.update_layout(title_text='Status per Subdistrict')
fig_table_sub.show()
output_tables['status_per_subdistrict_csv'] = subdistrict_status_table
output_figures['status_per_subdistrict_table_html'] = fig_table_sub


--- Table: Status per Subdistrict ---
         DISTRICT             SUBDISTRICT       STATUS
0       Abdiaziiz              Dhagaxtuur         Done
1       Abdiaziiz                 Gaarisa         Done
2       Abdiaziiz              Looyacadde         Done
3       Abdiaziiz                   Neero         Done
4       Boondheer                Daljirka         Done
5       Boondheer             Nasiibuundo         Done
6       Boondheer                 Siinaay         Done
7       Boondheer       Yuusuf Al-kowneyn         Done
8     Daarusalaam              1da Luulyo  Not Started
9     Daarusalaam             Ceel mareer  Not Started
10    Daarusalaam                  Labiga  Not Started
11    Daarusalaam               Raliweyne  Not Started
12    Daarusalaam                 Tawakal  Not Started
13       Dayniile                Barwaaqo  Not Started
14       Dayniile             Ciisa Cabdi  Not Started
15       Dayniile              Daarusalam  Not Started
16       Dayniile         

In [7]:
# 3. Table showing status per subdistrict
print("\n--- Table: Status per Subdistrict ---")
subdistrict_status_table = df[['DISTRICT', 'SUBDISTRICT', 'STATUS']].sort_values(by=['DISTRICT', 'SUBDISTRICT']).reset_index(drop=True)
print(subdistrict_status_table.to_string())

# Display as an interactive Plotly table
fig_table_sub = go.Figure(data=[go.Table(
    header=dict(values=list(subdistrict_status_table.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[subdistrict_status_table.DISTRICT, subdistrict_status_table.SUBDISTRICT, subdistrict_status_table.STATUS],
               fill_color='lavender',
               align='left'))
])
fig_table_sub.update_layout(title_text='Status per Subdistrict')
fig_table_sub.show()
output_tables['status_per_subdistrict_csv'] = subdistrict_status_table
output_figures['status_per_subdistrict_table_html'] = fig_table_sub


--- Table: Status per Subdistrict ---
         DISTRICT             SUBDISTRICT       STATUS
0       Abdiaziiz              Dhagaxtuur         Done
1       Abdiaziiz                 Gaarisa         Done
2       Abdiaziiz              Looyacadde         Done
3       Abdiaziiz                   Neero         Done
4       Boondheer                Daljirka         Done
5       Boondheer             Nasiibuundo         Done
6       Boondheer                 Siinaay         Done
7       Boondheer       Yuusuf Al-kowneyn         Done
8     Daarusalaam              1da Luulyo  Not Started
9     Daarusalaam             Ceel mareer  Not Started
10    Daarusalaam                  Labiga  Not Started
11    Daarusalaam               Raliweyne  Not Started
12    Daarusalaam                 Tawakal  Not Started
13       Dayniile                Barwaaqo  Not Started
14       Dayniile             Ciisa Cabdi  Not Started
15       Dayniile              Daarusalam  Not Started
16       Dayniile         

In [8]:
# 4. Bar graph of number of digitized buildings per district
print("\n--- Bar Graph: Total Digitized Buildings per District ---")
buildings_per_district = df.groupby('DISTRICT')['TOTAL_DONE'].sum().reset_index()
fig_buildings_district = px.bar(buildings_per_district,
                                x='DISTRICT',
                                y='TOTAL_DONE',
                                title='Total Digitized Buildings per District',
                                labels={'TOTAL_DONE': 'Total Buildings Digitized', 'DISTRICT': 'DISTRICT'},
                                color='TOTAL_DONE',
                                color_continuous_scale=px.colors.sequential.Plasma,
                                text='TOTAL_DONE')
fig_buildings_district.update_layout(xaxis={'categoryorder':'total descending'})
fig_buildings_district.show()
output_figures['total_buildings_per_district'] = fig_buildings_district


--- Bar Graph: Total Digitized Buildings per District ---


In [9]:
# 5. Pie chart of status in percentage
print("\n--- Pie Chart: Overall Status Distribution ---")
status_counts_percentage = df['STATUS'].value_counts(normalize=True).reset_index()
status_counts_percentage.columns = ['STATUS', 'Percentage']
status_counts_percentage['Percentage'] = status_counts_percentage['Percentage'] * 100

fig_pie_status = px.pie(status_counts_percentage,
                        values='Percentage',
                        names='STATUS',
                        title='Overall Status Distribution',
                        color='STATUS',
                        color_discrete_map=status_colors,
                        hole=0.3)
fig_pie_status.update_traces(textinfo='percent+label')
fig_pie_status.show()
output_figures['overall_status_distribution_pie'] = fig_pie_status


--- Pie Chart: Overall Status Distribution ---


In [10]:
# 6. Bar graph showing number of buildings per person assigned
print("\n--- Bar Graph: Total Digitized Buildings per Person Assigned ---")
buildings_per_person = df.groupby('Assigned')['TOTAL_DONE'].sum().reset_index()
fig_buildings_person = px.bar(buildings_per_person,
                              x='Assigned',
                              y='TOTAL_DONE',
                              title='Total Digitized Buildings per Person Assigned',
                              labels={'TOTAL_DONE': 'Total Buildings Digitized', 'Assigned': 'Person Assigned'},
                              color='TOTAL_DONE',
                              color_continuous_scale=px.colors.sequential.Viridis,
                              text='TOTAL_DONE')
fig_buildings_person.update_layout(xaxis={'categoryorder':'total descending'})
fig_buildings_person.show()
output_figures['total_buildings_per_person'] = fig_buildings_person


--- Bar Graph: Total Digitized Buildings per Person Assigned ---


In [11]:
# 7. Table showing total number of buildings done per person assigned
print("\n--- Table: Total Digitized Buildings per Person Assigned ---")
buildings_per_person_table = df.groupby('Assigned')['TOTAL_DONE'].sum().reset_index()
buildings_per_person_table = buildings_per_person_table.sort_values(by='TOTAL_DONE', ascending=False).reset_index(drop=True)
print(buildings_per_person_table.to_string())

# Display as an interactive Plotly table
fig_table_person = go.Figure(data=[go.Table(
    header=dict(values=list(buildings_per_person_table.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[buildings_per_person_table.Assigned, buildings_per_person_table.TOTAL_DONE],
               fill_color='lavender',
               align='left'))
])
fig_table_person.update_layout(title_text='Total Digitized Buildings per Person Assigned')
fig_table_person.show()
output_tables['total_buildings_per_person_table_csv'] = buildings_per_person_table
output_figures['total_buildings_per_person_table_html'] = fig_table_person


--- Table: Total Digitized Buildings per Person Assigned ---
     Assigned  TOTAL_DONE
0  Abdisalaan     24377.0
1     Sabrina     15571.0
2     Sumaiya     15333.0
3  Abdirahman     10144.0
4      Ridwan      7866.0


In [12]:
# 8. Total number of buildings done (overall)
print("\n--- Total Number of Buildings Done (Overall) ---")
total_overall_buildings_done = df['TOTAL_DONE'].sum()
print(f"The grand total number of buildings digitized across all districts is: {int(total_overall_buildings_done)}")
final_summaries['total_overall_buildings_done'] = int(total_overall_buildings_done)


--- Total Number of Buildings Done (Overall) ---
The grand total number of buildings digitized across all districts is: 86759


In [ ]:
# 9. Export All Outputs
print("\n--- 9. Exporting All Outputs ---")
output_dir = "analysis_outputs"
os.makedirs(output_dir, exist_ok=True)
print(f"Saving outputs to '{output_dir}' directory.")

# Export Plotly figures as HTML
for name, fig in output_figures.items():
    file_path = os.path.join(output_dir, f"{name}.html")
    fig.write_html(file_path)
    print(f"  - Exported {name}.html")

# Export pandas DataFrames as CSV
for name, df_to_save in output_tables.items():
    file_path = os.path.join(output_dir, f"{name}.csv")
    df_to_save.to_csv(file_path, index=False)
    print(f"  - Exported {name}.csv")

# Export final summaries
for name, value in final_summaries.items():
    file_path = os.path.join(output_dir, f"{name}.txt")
    with open(file_path, "w") as f:
        f.write(f"{name.replace('_', ' ').title()}: {value}")
    print(f"  - Exported {name}.txt")

# Zip the output directory for easy download
print("\n--- Zipping output files for download ---")
zip_file_name = "analysis_outputs.zip"
!zip -r /content/{zip_file_name} /content/{output_dir} # Corrected path for zipping
files.download(zip_file_name)
print(f"\nAll outputs zipped and ready for download as '{zip_file_name}'!")

In [ ]:
# 10. Create and Export a Combined PDF Report
print("\n--- 10. Creating and Exporting Combined PDF Report ---")
pdf_output_dir = "pdf_report"
os.makedirs(pdf_output_dir, exist_ok=True)
pdf_file_name = os.path.join(pdf_output_dir, "Status_Report_Analysis.pdf")
html_report_path = os.path.join(pdf_output_dir, "report.html")

# Prepare data for the HTML template
report_data = {
    'title': 'Status Report Analysis',
    'sections': []
}

# --- Add Summaries to Report Data ---
summary_html_content = "<h2>Overall Summary</h2>"
for name, value in final_summaries.items():
    summary_html_content += f"<p><strong>{name.replace('_', ' ').title()}:</strong> {value}</p>"
report_data['sections'].append({
    'title': 'Overall Summary',
    'content_type': 'html',
    'content': summary_html_content
})

# --- Convert Plots to PNG and add to Report Data ---
print("Converting plots to static images for PDF...")
for name, fig in output_figures.items():
    # Only convert the actual chart figures, not the Plotly table figures for the PDF
    # We will represent tables directly as HTML in the PDF
    if "table" not in name:
        try:
            # Export Plotly figure as a static PNG image
            img_bytes = fig.to_image(format="png", engine="kaleido", scale=2) # scale for better resolution
            img_base64 = base64.b64encode(img_bytes).decode('utf-8')
            report_data['sections'].append({
                'title': fig.layout.title.text if fig.layout.title else name.replace('_', ' ').title(),
                'content_type': 'image',
                'content': f"data:image/png;base64,{img_base64}"
            })
            print(f"  - Converted plot '{name}' to PNG.")
        except Exception as e:
            print(f"  - Failed to convert plot '{name}' to PNG: {e}")

# --- Add Tables (as HTML strings) to Report Data ---
print("Converting tables to HTML for PDF...")
for name, df_to_save in output_tables.items():
    html_table = df_to_save.to_html(index=False, classes='dataframe table table-striped')
    report_data['sections'].append({
        'title': name.replace('_', ' ').title().replace('_Csv', ''), # Clean up name for display
        'content_type': 'html',
        'content': f"<h2>{name.replace('_', ' ').title().replace('_Csv', '')}</h2>{html_table}"
    })
    print(f"  - Converted table '{name}' to HTML.")

# --- Generate HTML Report ---
# Create a simple Jinja2 template for the PDF report structure
html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>{{ title }}</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20mm; }
        h1 { text-align: center; color: #333; }
        h2 { border-bottom: 1px solid #ccc; padding-bottom: 5px; margin-top: 30px; color: #555; }
        img { max-width: 100%; height: auto; display: block; margin: 10px auto; }
        table { width: 100%; border-collapse: collapse; margin-top: 10px; }
        th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
        th { background-color: #f2f2f2; }
        .page-break { page-break-before: always; }
    </style>
</head>
<body>
    <h1>{{ title }}</h1>
    {% for section in sections %}
        <div class="page-break">
            {% if section.content_type == 'image' %}
                <h2>{{ section.title }}</h2>
                <img src="{{ section.content }}" alt="{{ section.title }}">
            {% elif section.content_type == 'html' %}
                {{ section.content | safe }}
            {% endif %}
        </div>
    {% endfor %}
</body>
</html>
"""

# Render the HTML report
env = Environment(loader=FileSystemLoader('.')) # Use current directory for template
template = env.from_string(html_template)
rendered_html = template.render(report_data)

with open(html_report_path, "w", encoding="utf-8") as f:
    f.write(rendered_html)
print(f"Generated HTML report at '{html_report_path}'")

# --- Convert HTML to PDF using WeasyPrint ---
from weasyprint import HTML, CSS
try:
    HTML(string=rendered_html).write_pdf(pdf_file_name, stylesheets=[CSS(string='@page { size: A4; margin: 1cm; }')])
    print(f"PDF report created successfully at '{pdf_file_name}'")
except Exception as e:
    print(f"Error creating PDF: {e}")
    print("Please check if weasyprint and its dependencies are correctly installed.")


# Download the PDF
files.download(pdf_file_name)
print(f"\nPDF report '{pdf_file_name}' downloaded successfully!")